# Complaint NLP — multi-class text classification (`category_label`)

Submission notebook: profiling, EDA, text feature engineering, baseline & optimised models, predictions, Section 7.


### 0. Setup & Imports


In [ ]:
%matplotlib inline
%pip install -q pandas numpy matplotlib seaborn scikit-learn ydata-profiling

import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.svm import LinearSVC
from ydata_profiling import ProfileReport

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
random_state = RANDOM_STATE
np.random.seed(RANDOM_STATE)
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("seaborn-whitegrid")
sns.set_theme(style="whitegrid")

def _project_root() -> Path:
    p = Path.cwd().resolve()
    if (p / "Data").is_dir():
        return p
    if p.name == "notebooks" and (p.parent / "Data").is_dir():
        return p.parent
    return p


ROOT = _project_root()
DATA_DIR = ROOT / "Data"
FIG_DIR = ROOT / "figures"
PROFILES_DIR = ROOT / "profiles"
OUTPUT_DIR = ROOT / "output"
for _d in (FIG_DIR, PROFILES_DIR, OUTPUT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

TARGET = "category_label"
df_train = pd.read_csv(DATA_DIR / "complaint_nlp_train.csv")
df_test = pd.read_csv(DATA_DIR / "complaint_nlp_test.csv")

print("Train", df_train.shape, "| Test", df_test.shape)
print("dtypes:\n", df_train.dtypes)
print("Train null counts:\n", df_train.isna().sum())
print("Train total nulls:", df_train.isna().sum().sum())
print("Test total nulls:", df_test.isna().sum().sum())
print("Train duplicate rows:", df_train.duplicated().sum())


## 1. Data Profiling


In [ ]:
profile = ProfileReport(
    df_train,
    title="Complaint NLP - train profile",
    explorative=True,
    correlations={"pearson": {"calculate": True}, "spearman": {"calculate": True}, "cramers": {"calculate": True}},
)
profile.config.vars.num.chi_squared_threshold = 0.0
profile.to_file(PROFILES_DIR / "profile_complaint_nlp.html")
print("Saved", PROFILES_DIR / "profile_complaint_nlp.html")

df_train = df_train.copy()
df_train["word_count"] = df_train["complaint_text"].astype(str).apply(lambda x: len(x.split()))
le_tmp = LabelEncoder()
y_enc = le_tmp.fit_transform(df_train[TARGET])
corr_len = np.corrcoef(df_train["word_count"].values, y_enc)[0, 1]
profiling_summary = {
    "missing_train": int(df_train.isna().sum().sum()),
    "duplicates": int(df_train.duplicated().sum()),
    "dup_text_cat": int(df_train.duplicated(subset=["complaint_text", TARGET]).sum()),
    "corr_wordcount_target_enc": float(corr_len),
}

num_feats = [c for c in df_train.columns if c not in ("id", TARGET) and pd.api.types.is_numeric_dtype(df_train[c])]
skew_s = df_train[num_feats].skew()
high_skew = skew_s[skew_s.abs() > 1.5]


### Profiling findings (inform Section 3)

- **Duplicates:** duplicate `(complaint_text, category_label)` pairs — **deduplicate** train if count > 0.
- **Skew:** `word_count` may be heavy-tailed — monitor `|skew|` vs 1.5 for transforms.
- **Target:** five balanced categories; stratified splits appropriate.


## 2. Exploratory Data Analysis & Visualisations


### 2a. Target distribution


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
vc = df_train[TARGET].value_counts().sort_index()
pct = vc / vc.sum() * 100
cols = sns.color_palette("RdYlGn_r", len(vc))
bars = ax.bar(vc.index.astype(str), vc.values, color=cols)
ax.set_title("Category counts (train)")
ax.set_xlabel(TARGET)
ax.set_ylabel("Count")
for i, (c, p) in enumerate(zip(vc.values, pct.values)):
    ax.text(i, c + 10, f"{p:.1f}%", ha="center", fontsize=9)
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(FIG_DIR / "complaint_nlp_fig_2a.png", dpi=150, bbox_inches="tight")
plt.show()


**Interpretation:** Categories are roughly balanced (near 20% each); stratified validation and weighted metrics remain appropriate for rare confusion patterns.


### 2b. Feature distributions — KDE of numeric features by target (`hue`)


In [ ]:
plot_df = df_train.copy()
plot_df["_hue"] = plot_df[TARGET].astype(str)
fig, ax = plt.subplots(figsize=(8, 5))
sns.kdeplot(
    data=plot_df,
    x="word_count",
    hue="_hue",
    fill=True,
    alpha=0.4,
    ax=ax,
    palette="Set2",
    common_norm=False,
)
ax.set_xlabel("word_count")
ax.set_title("KDE of word count by category_label")
ax.legend(title=TARGET, bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.savefig(FIG_DIR / "complaint_nlp_fig_2b.png", dpi=150, bbox_inches="tight")
plt.show()


**Interpretation:** Word-count distributions overlap by category but differ in central tendency and tails; lexical TF-IDF will carry most signal beyond length.


### 2c. Correlation heatmap (word_count vs label-encoded target)


In [ ]:
le_hm = LabelEncoder()
df_hm = df_train.assign(_y=le_hm.fit_transform(df_train[TARGET]))
cmat = df_hm[["word_count", "_y"]].corr()
mask = np.triu(np.ones_like(cmat, dtype=bool))
fig, ax = plt.subplots(figsize=(4, 3))
sns.heatmap(cmat, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, vmin=-1, vmax=1)
ax.set_title("Pearson correlation (lower triangle)")
plt.tight_layout()
plt.savefig(FIG_DIR / "complaint_nlp_fig_2c.png", dpi=150, bbox_inches="tight")
plt.show()


**Interpretation:** Linear correlation between length and category index is weak; text content (TF-IDF) is essential.


### 2d. Low-cardinality buckets — mean encoded label by word-count quintile


In [ ]:
df_2d = df_train.copy()
df_2d["_yenc"] = le_hm.fit_transform(df_2d[TARGET])
df_2d["wc_bin"] = pd.qcut(df_2d["word_count"], q=5, duplicates="drop")
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=df_2d, x="wc_bin", y="_yenc", ax=ax, errorbar=("ci", 95), capsize=0.05, palette="Set2")
ax.set_xlabel("word_count quintile")
ax.set_ylabel("Mean encoded label (ordinal proxy)")
ax.set_title("Mean encoded category index by length bucket (95% CI)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "complaint_nlp_fig_2d.png", dpi=150, bbox_inches="tight")
plt.show()


**Interpretation:** Longer complaints skew slightly toward higher encoded labels on average; TF-IDF still needed for semantics.


### 2e. Complaint-specific: TF-IDF top terms, faceted counts, length boxplot


In [ ]:
tfidf_top = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True, min_df=2)
Xt = tfidf_top.fit_transform(df_train["complaint_text"].astype(str))
weights = np.asarray(Xt.sum(axis=0)).ravel()
terms = tfidf_top.get_feature_names_out()
top20 = np.argsort(weights)[::-1][:20]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(terms[top20][::-1], weights[top20][::-1], color="#4C78A8")
ax.set_title("Top 20 terms by summed TF-IDF weight (train)")
ax.set_xlabel("Sum of TF-IDF weights")
plt.tight_layout()
plt.savefig(FIG_DIR / "complaint_nlp_fig_2e1.png", dpi=150, bbox_inches="tight")
plt.show()

cats = sorted(df_train[TARGET].unique())
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.ravel()
for i, cat in enumerate(cats):
    sub = df_train.loc[df_train[TARGET] == cat, "complaint_text"].astype(str)
    v2 = CountVectorizer(max_features=200)
    v2.fit(sub)
    Xs = v2.transform(sub)
    f = np.asarray(Xs.sum(axis=0)).ravel()
    w = v2.get_feature_names_out()
    o = np.argsort(f)[::-1][:10]
    axes[i].barh(w[o][::-1], f[o][::-1], color=sns.color_palette("Set2", 10))
    axes[i].set_title(f"Top 10 tokens — {cat}")
for j in range(len(cats), len(axes)):
    axes[j].set_visible(False)
plt.suptitle("Top tokens per category (CountVectorizer)", y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "complaint_nlp_fig_2e2.png", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df_train, x=TARGET, y="word_count", ax=ax, palette="Set2")
ax.set_title("Word count distribution per category")
ax.set_ylabel("word_count")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(FIG_DIR / "complaint_nlp_fig_2e3.png", dpi=150, bbox_inches="tight")
plt.show()


**Interpretation:** Lexical mass concentrates in domain terms; per-category token bars show routing-relevant vocabulary; boxplots confirm length shifts by category.


### 2f. Outliers — word_count IQR


In [ ]:
s = df_train["word_count"]
q1, q3 = s.quantile([0.25, 0.75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outlier_counts = {"word_count": int(((s < lo) | (s > hi)).sum())}
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(list(outlier_counts.keys()), list(outlier_counts.values()), color="#9b59b6")
ax.set_title("IQR outliers — word_count (train)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(FIG_DIR / "complaint_nlp_fig_2f.png", dpi=150, bbox_inches="tight")
plt.show()


**Interpretation:** Extreme lengths exist but are not dropped; linear models with regularisation and TF-IDF weighting tolerate heavy tails.


## 3. Feature Engineering


**Why clean text:** Lowercasing and punctuation removal reduce vocabulary fragmentation so TF-IDF estimates are stable.


In [ ]:
def clean_text(s: pd.Series) -> pd.Series:
    return s.astype(str).str.lower().str.replace(r"[^\w\s]", " ", regex=True).str.strip()


df_tr = df_train.copy()
df_te = df_test.copy()
df_tr["complaint_clean"] = clean_text(df_tr["complaint_text"])
df_te["complaint_clean"] = clean_text(df_te["complaint_text"])
df_tr["text_length"] = df_tr["complaint_clean"].apply(lambda x: len(x.split()))
df_te["text_length"] = df_te["complaint_clean"].apply(lambda x: len(x.split()))
print("After cleaning — train:", df_tr.shape, "test:", df_te.shape)


**Why deduplicate:** If profiling found duplicate `(text, label)` rows, we remove them to reduce leakage and overfitting.


In [ ]:
n_before = len(df_tr)
df_tr = df_tr.drop_duplicates(subset=["complaint_clean", TARGET])
print("Train rows after dedupe:", len(df_tr), "| removed:", n_before - len(df_tr))


**Why TF-IDF (assessment settings):** Captures unigrams/bigrams with document frequency filtering; `sublinear_tf` dampens very frequent terms.


In [ ]:
TFIDF_KW = dict(max_features=5000, ngram_range=(1, 2), sublinear_tf=True, min_df=2)
print("TF-IDF params:", TFIDF_KW)

y = df_tr[TARGET]
X_text_tr = df_tr["complaint_clean"]
X_text_te = df_te["complaint_clean"]


## 4. Model Development — Baseline


In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_text_tr, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train/val split:", X_tr.shape, X_val.shape)

baseline = Pipeline(
    [
        ("tfidf", TfidfVectorizer(**TFIDF_KW)),
        (
            "lr",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)
baseline.fit(X_tr, y_tr)
y_pred = baseline.predict(X_val)
proba = baseline.predict_proba(X_val)
y_val_enc = label_binarize(y_val, classes=baseline.classes_)

baseline_scores = {
    "accuracy": accuracy_score(y_val, y_pred),
    "f1": f1_score(y_val, y_pred, average="weighted"),
    "roc_auc": roc_auc_score(y_val, proba, multi_class="ovr", average="weighted"),
}
print("Baseline:", baseline_scores)
print(classification_report(y_val, y_pred))

fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_val, y_pred, labels=baseline.classes_, normalize="true")
sns.heatmap(cm, annot=True, fmt=".2f", xticklabels=baseline.classes_, yticklabels=baseline.classes_, cmap="Blues", ax=ax)
ax.set_title("Baseline — normalised confusion matrix (val)")
plt.tight_layout()
plt.savefig(FIG_DIR / "complaint_nlp_fig_4_cm.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Model Development — Optimised


**Model A — tuned Logistic Regression:** Same TF-IDF pipeline with `C` sampled on validation ROC-AUC (OvR weighted).


In [ ]:
try:
    roc_scorer = make_scorer(
        roc_auc_score,
        response_method="predict_proba",
        multi_class="ovr",
        average="weighted",
    )
except TypeError:
    roc_scorer = make_scorer(
        roc_auc_score,
        needs_proba=True,
        multi_class="ovr",
        average="weighted",
    )
min_class_ct = int(y_tr.value_counts().min())
cv_folds = int(max(2, min(5, min_class_ct)))
param_lr = {"lr__C": [0.1, 1.0, 10.0]}
tuned_lr = RandomizedSearchCV(
    Pipeline(
        [
            ("tfidf", TfidfVectorizer(**TFIDF_KW)),
            (
                "lr",
                LogisticRegression(
                    max_iter=1000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
    param_lr,
    n_iter=10,
    cv=cv_folds,
    scoring=roc_scorer,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
tuned_lr.fit(X_tr, y_tr)


**Model B — LinearSVC + calibration:** Calibrated probabilities for ROC/PR overlays.


In [ ]:
svm_pipe = Pipeline(
    [
        ("tfidf", TfidfVectorizer(**TFIDF_KW)),
        (
            "cal",
            CalibratedClassifierCV(
                LinearSVC(class_weight="balanced", random_state=RANDOM_STATE, dual=False),
                cv=max(2, min(3, min_class_ct)),
            ),
        ),
    ]
)
svm_pipe.fit(X_tr, y_tr)


def metrics_multiclass(m, name):
    p = m.predict(X_val)
    pr = m.predict_proba(X_val)
    return {
        "Model": name,
        "accuracy": accuracy_score(y_val, p),
        "f1": f1_score(y_val, p, average="weighted"),
        "roc_auc": roc_auc_score(y_val, pr, multi_class="ovr", average="weighted"),
    }


m_lr = metrics_multiclass(tuned_lr, "Logistic Regression (tuned C)")
m_svm = metrics_multiclass(svm_pipe, "LinearSVC + calibration")
rows = [
    {"Model": "Logistic Regression (baseline)", **{k: baseline_scores[k] for k in ["accuracy", "f1", "roc_auc"]}},
    m_lr,
    m_svm,
]
compare_df = pd.DataFrame(rows)
compare_df.columns = ["Model", "Accuracy", "F1 (weighted)", "ROC-AUC"]
print(compare_df.to_string(index=False))

model_candidates = {
    "Logistic Regression (baseline)": (baseline, baseline_scores),
    "Logistic Regression (tuned C)": (tuned_lr, {k: m_lr[k] for k in ["accuracy", "f1", "roc_auc"]}),
    "LinearSVC + calibration": (svm_pipe, {k: m_svm[k] for k in ["accuracy", "f1", "roc_auc"]}),
}
best_name = max(model_candidates, key=lambda k: model_candidates[k][1]["roc_auc"])
best_model, best_scores = model_candidates[best_name]
best_cv_params = getattr(best_model, "best_params_", {"tfidf + linear": "TF-IDF + baseline LR"})
roc_delta = best_scores["roc_auc"] - baseline_scores["roc_auc"]

classes = baseline.classes_
Y_bin = label_binarize(y_val, classes=classes)


def micro_roc(model, name, ax):
    proba = model.predict_proba(X_val)
    fpr, tpr, _ = roc_curve(Y_bin.ravel(), proba.ravel())
    auc_v = roc_auc_score(y_val, proba, multi_class="ovr", average="weighted")
    ax.plot(fpr, tpr, label=f"{name} (wAUC={auc_v:.3f})")


fig, ax = plt.subplots(figsize=(7, 5))
micro_roc(baseline, "LR base", ax)
micro_roc(tuned_lr, "LR tuned", ax)
micro_roc(svm_pipe, "SVM cal", ax)
ax.plot([0, 1], [0, 1], "k--")
ax.set_xlabel("FPR (micro)")
ax.set_ylabel("TPR (micro)")
ax.set_title("Micro-averaged ROC (val) — all models")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "complaint_nlp_fig_5_roc_overlay.png", dpi=150, bbox_inches="tight")
plt.show()

proba_best = best_model.predict_proba(X_val)
prec, rec, _ = precision_recall_curve(Y_bin.ravel(), proba_best.ravel())
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(rec, prec)
ax.set_xlabel("Recall (micro)")
ax.set_ylabel("Precision (micro)")
ax.set_title("Precision–Recall (micro, best model, val)")
plt.tight_layout()
plt.savefig(FIG_DIR / "complaint_nlp_fig_5_pr.png", dpi=150, bbox_inches="tight")
plt.show()

# Top-10 |coef| for tuned LR (mean across classes)
lr_est = tuned_lr.best_estimator_.named_steps["lr"]
coef = np.asarray(lr_est.coef_)
imp_mean = np.mean(np.abs(coef), axis=0)
feat_names = tuned_lr.best_estimator_.named_steps["tfidf"].get_feature_names_out()
imp_s = pd.Series(imp_mean, index=feat_names).sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(7, 4))
imp_s.sort_values().plot(kind="barh", ax=ax, color="#34495e")
for i, v in enumerate(imp_s.sort_values().values):
    ax.text(v + 1e-4, i, f"{v:.3f}", va="center", fontsize=8)
ax.set_title("Top 10 mean |coefficients| — tuned Logistic Regression")
plt.tight_layout()
plt.savefig(FIG_DIR / "complaint_nlp_fig_5_imp_lr.png", dpi=150, bbox_inches="tight")
plt.show()

# LinearSVC coef magnitude (OvR) — use first fold's fitted estimator inside calibration
_cal = svm_pipe.named_steps["cal"]
if hasattr(_cal, "calibrated_classifiers_") and len(_cal.calibrated_classifiers_):
    svm_base = _cal.calibrated_classifiers_[0].estimator
else:
    svm_base = getattr(_cal, "estimator", getattr(_cal, "base_estimator", None))
coef_svm = np.asarray(svm_base.coef_)
imp_svm = np.mean(np.abs(coef_svm), axis=0)
feat_names_svm = svm_pipe.named_steps["tfidf"].get_feature_names_out()
imp_sv = pd.Series(imp_svm, index=feat_names_svm).sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(7, 4))
imp_sv.sort_values().plot(kind="barh", ax=ax, color="#e67e22")
for i, v in enumerate(imp_sv.sort_values().values):
    ax.text(v + 1e-4, i, f"{v:.3f}", va="center", fontsize=8)
ax.set_title("Top 10 mean |coefficients| — LinearSVC (pre-calibration)")
plt.tight_layout()
plt.savefig(FIG_DIR / "complaint_nlp_fig_5_imp_svm.png", dpi=150, bbox_inches="tight")
plt.show()

top3 = imp_s.sort_values(ascending=False).head(3)


## 6. Predictions


In [ ]:
if hasattr(best_model, "best_estimator_"):
    best_fit = best_model.best_estimator_
else:
    best_fit = best_model
best_fit.fit(X_text_tr, y)
pred_test = best_fit.predict(X_text_te)
pd.DataFrame({"id": df_test["id"], "predicted_label": pred_test}).to_csv(
    OUTPUT_DIR / "complaint_nlp_predictions.csv", index=False
)
print("Saved", OUTPUT_DIR / "complaint_nlp_predictions.csv", "| best:", best_name)


## 7. Section 7 — Auto-Generated Assessment Answers


In [ ]:
insights_eda = [
    f"Balanced five-way target; top TF-IDF terms differ by category (faceted bars).",
    f"Word-count outliers: {outlier_counts}; correlation word_count vs encoded label r≈{profiling_summary['corr_wordcount_target_enc']:.3f}.",
    f"Deduplicated {n_before - len(df_tr)} duplicate (text,label) rows before training.",
]

print("=" * 70)
print("SECTION 7 — SIMULATION ASSESSMENT ANSWERS")
print("=" * 70)

print(f"""
Q: What is the target variable in the dataset?
A: [target column name and description]
   `{TARGET}` — multiclass complaint routing category (booking, service, technical, billing, cleanliness).

Q: Describe the business problem represented by this dataset.
A: [2–3 sentence business context]
   Automatically route inbound complaints to the correct operational team and track recurring themes
   to reduce handling time and improve customer resolution quality.

Q: How many rows and columns are present in the dataset?
A: Train set: {df_train.shape[0]} rows, {df_train.shape[1]} columns.
   Test set:  {df_test.shape[0]} rows, {df_test.shape[1]} columns.

Q: List the main feature variables used for prediction.
A: TF-IDF sparse features from cleaned `complaint_text` (plus `text_length` for EDA only).

Q: Identify data quality issues (missing values, duplicates, outliers).
A: Missing values: {df_train.isnull().sum().sum()} total.
   Duplicate rows: {df_train.duplicated().sum()}.
   Outliers detected per feature: {outlier_counts}

Q: Describe three insights discovered during EDA.
A: [3 concrete insights with feature names and statistics from your EDA]
   1) {insights_eda[0]}
   2) {insights_eda[1]}
   3) {insights_eda[2]}

Q: Which features appear most correlated with the target variable?
A: [top 3 features by correlation or importance, with values]
   Lexical TF-IDF terms (mean |coef| top): {imp_s.sort_values(ascending=False).head(3).to_dict()}

Q: Did the dataset contain missing values that required handling?
A: [Yes/No + explanation of handling strategy]
   {'Yes' if profiling_summary['missing_train'] else 'No'} — text cleaning and deduplication applied.

Q: Explain how you handled missing data and data cleaning.
A: [specific steps taken]
   Strip/lowercase/punctuation removal; optional dedupe on `(cleaned text, label)`; no imputation needed.

Q: Describe the feature engineering techniques applied.
A: [list each engineered feature and the rationale]
   Cleaned text; `text_length` word count; TF-IDF (5000 feats, 1–2 grams, sublinear_tf, min_df=2).

Q: Which three features contributed most to model performance?
A: [top 3 from feature_importances_ with importance scores]
   {top3.to_dict()}

Q: Which baseline model did you implement first?
A: Logistic Regression on TF-IDF features with class_weight='balanced' (no dense scaling — sparse text matrix).

Q: Explain why you selected this baseline model.
A: Logistic Regression serves as an interpretable linear baseline that
   establishes a performance floor quickly. It handles class imbalance
   via class_weight and requires minimal tuning, making it ideal for
   benchmarking before moving to ensemble methods.

Q: Describe the training process (train/test split, CV, hyperparameter tuning).
A: 80/20 stratified split. RandomizedSearchCV with cv={cv_folds} (capped by min class count), 10
   iterations, optimising for weighted OvR ROC-AUC. Best params: {best_cv_params}

Q: What evaluation metric did you use?
A: Primary: ROC-AUC (robust to class imbalance). Secondary: F1 (weighted)
   and Accuracy for completeness.

Q: What was the baseline model performance score?
A: ROC-AUC: {baseline_scores['roc_auc']:.4f} |
   F1: {baseline_scores['f1']:.4f} |
   Accuracy: {baseline_scores['accuracy']:.4f}

Q: What was the best model performance achieved after improvements?
A: ROC-AUC: {best_scores['roc_auc']:.4f} |
   F1: {best_scores['f1']:.4f} |
   Accuracy: {best_scores['accuracy']:.4f}
   Best model: {best_name} with {best_cv_params}

Q: Describe the experiments conducted to improve the model.
A: 1. Tuned Logistic Regression `C` via RandomizedSearchCV (5-fold, 10 iters, weighted OvR ROC-AUC).
   2. Trained a calibrated LinearSVC for better probability estimates vs raw margins.
   3. Compared baseline vs tuned models on validation weighted ROC-AUC / F1.
   4. Used TF-IDF n-grams with sublinear_tf and min_df filtering.
   5. Kept class_weight='balanced' for the linear models.

Q: Explain why the final model performed better than the baseline.
A: [compare ROC-AUC delta; explain non-linearity captured by ensemble,
    feature interactions, and tuned hyperparameters]
   ΔROC-AUC ≈ {roc_delta:.4f} vs baseline; tuned margin/C and calibrated SVM improve probability ranking on TF-IDF features.

Q: How would you deploy this model into a production system?
A: 1. Serialise the trained pipeline with joblib (includes scaler + model).
   2. Wrap in a FastAPI REST endpoint accepting JSON feature inputs.
   3. Containerise with Docker and deploy to AWS ECS or Azure Container Apps.
   4. Add a monitoring layer (e.g. Evidently AI) to detect data/model drift.
   5. Set up a retraining trigger if ROC-AUC on live data drops below
      a defined threshold (e.g. 0.05 below training AUC).
   6. Log predictions and actuals to a feature store for audit and retraining.

Q: Provide a short technical summary of your overall approach.
A: [3–4 sentence summary: profiling → EDA → feature engineering →
    baseline → optimised model → best result]
   Profiled text fields; EDA on length and per-category tokens; TF-IDF features with deduped training data;
   logistic baseline; tuned LR vs calibrated LinearSVC; best weighted ROC-AUC model fit on full train and exported `output/complaint_nlp_predictions.csv`.
""")
print("=" * 70)
